In [1]:
import json
import re
import fitz  # PyMuPDF
from typing import List, Dict, Optional, Tuple


# ============================================================
# 1. PDF EXTRACTION
# ============================================================
def extract_text_with_fitz(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    pages = [page.get_text("text") for page in doc]
    return "\n".join(pages)


# ============================================================
# 2. CLEANING – Fixes all issues found in original
# ============================================================
def clean_legal_pdf_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # --- Header/Footer ---
    # Original bug: used \+ (literal plus) instead of matching space-separated number
    text = re.sub(
        r"CÔNG BÁO/Số\s+\d+(?:\s*\+\s*\d+)?/Ngày\s+\d{1,2}-\d{1,2}-\d{4}\s*",
        " ", text, flags=re.IGNORECASE
    )

    # Remove isolated page numbers (standalone numbers on their own line)
    text = re.sub(r"\n\s*\d{1,3}\s*\n", "\n", text)

    # --- Footnote blocks at bottom of pages ---
    # Pattern: line starting with a number followed by "Luật An ninh mạng" or "Cụm từ" or "Điều XX của Luật"
    text = re.sub(
        r"\n\s*\d{1,2}\s+(Luật An ninh mạng|Cụm từ|Điều\s+\d+\s+của\s+Luật).+?(?=\n(?:Điều\s+\d|Chương\s+[IVX]|\d+\.\s|[a-gđ]\))|$)",
        "\n", text, flags=re.DOTALL
    )

    # --- Inline superscript footnote markers (e.g., "an ninh mạng²", "mạng,⁸") ---
    text = re.sub(r"[¹²³⁴⁵⁶⁷⁸⁹⁰]+", "", text)
    # Also handle regular digits used as inline footnote refs (e.g., "mạng2 của")
    # These appear as digit right after a word with no space
    text = re.sub(r'([^\d\s])(\d{1,2})(\s+(?:của|theo|giao|và|hoặc|trong))', r'\1\3', text)

    # --- Preamble before Chương I (law title, citation info) ---
    # We keep this – it provides useful context for the first chunk

    # --- Trailing signature block ---
    text = re.sub(
        r"VĂN PHÒNG QUỐC HỘI.*$", "", text, flags=re.DOTALL
    )

    # --- Whitespace normalization ---
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"^\s+", "", text, flags=re.MULTILINE)

    return text.strip()


# ============================================================
# 3. STRUCTURAL PARSING: Chương → Mục → Điều
# ============================================================

CHAPTER_RE = re.compile(
    r"(Chương\s+[IVXLCDM]+)\s*\n\s*([A-ZĐÀ-Ỹ][^\n]+)",
    flags=re.UNICODE
)

SECTION_RE = re.compile(
    r"(Mục\s+\d+)\s*\n\s*([A-ZĐÀ-Ỹ][^\n]+)",
    flags=re.UNICODE
)

ARTICLE_RE = re.compile(
    r"(Điều\s+\d+[a-z]?\.\s*[^\n]+)",
    flags=re.UNICODE
)

# Khoản-level split points: "1. ", "2. ", etc. at start of line
KHOAN_RE = re.compile(r"\n(?=\d{1,2}\.\s)")


def _find_spans(pattern, text) -> List[Tuple[int, int, str, str]]:
    """Return list of (start, end_of_match, label, title) for a heading pattern."""
    results = []
    for m in pattern.finditer(text):
        results.append((m.start(), m.end(), m.group(1).strip(), m.group(2).strip()))
    return results


def parse_structure(text: str) -> List[Dict]:
    """
    Parse Vietnamese legal text into a flat list of articles,
    each annotated with its Chương and Mục (if any).
    """
    chapters = _find_spans(CHAPTER_RE, text)
    sections = _find_spans(SECTION_RE, text)
    article_matches = list(ARTICLE_RE.finditer(text))

    def _find_enclosing(pos: int, spans: list) -> Optional[str]:
        """Find which chapter/section a position falls in."""
        result = None
        for start, end, label, title in spans:
            if start <= pos:
                result = f"{label} {title}"
            else:
                break
        return result

    articles = []
    for i, m in enumerate(article_matches):
        art_start = m.start()
        art_end = article_matches[i + 1].start() if i + 1 < len(article_matches) else len(text)

        article_title = m.group(1).strip()
        article_body = text[m.end():art_end].strip()

        chapter_full = _find_enclosing(art_start, chapters)
        section_full = _find_enclosing(art_start, sections)

        articles.append({
            "article_title": article_title,
            "article_body": article_body,
            "chapter": chapter_full,
            "section": section_full,  # Mục level – new!
        })

    return articles


# ============================================================
# 4. SMART SPLITTING & MERGING
# ============================================================

def split_article_by_khoan(title: str, body: str, max_len: int) -> List[str]:
    """
    Split a long article on khoản (clause) boundaries.
    Each sub-chunk gets the article title prepended for retrieval context.
    """
    full_text = f"{title}\n{body}"

    if len(full_text) <= max_len:
        return [full_text]

    # Split on khoản boundaries
    parts = KHOAN_RE.split(body)
    parts = [p.strip() for p in parts if p.strip()]

    chunks = []
    current = title  # Always start with the article title

    for part in parts:
        candidate = f"{current}\n{part}" if current != title else f"{title}\n{part}"

        if len(candidate) <= max_len:
            current = candidate
        else:
            # Flush current chunk if it has content beyond just the title
            if current != title:
                chunks.append(current.strip())

            # If this single part + title already exceeds max, do a hard split
            single = f"{title}\n{part}"
            if len(single) > max_len:
                # Hard split preserving title context
                offset = 0
                while offset < len(part):
                    # Find a good break point
                    end = min(offset + max_len - len(title) - 50, len(part))
                    if end < len(part):
                        # Try to break at sentence end
                        for sep in ["\n", ". ", "; ", ", "]:
                            last_sep = part.rfind(sep, offset, end)
                            if last_sep > offset:
                                end = last_sep + len(sep)
                                break
                    chunk_text = part[offset:end].strip()
                    if chunk_text:
                        chunks.append(f"{title}\n{chunk_text}")
                    offset = end
                current = title
            else:
                current = single

    if current != title:
        chunks.append(current.strip())

    return chunks if chunks else [full_text[:max_len]]


def merge_short_articles(articles: List[Dict], min_len: int = 400) -> List[Dict]:
    """
    Merge consecutive short articles (< min_len chars) that share the same
    Mục/Chapter into a single chunk. This prevents tiny orphan chunks that
    lack sufficient context for retrieval.
    """
    if not articles:
        return articles

    merged = []
    buffer = None

    for art in articles:
        body_len = len(art["article_body"])

        if body_len >= min_len:
            # Flush buffer
            if buffer:
                merged.append(buffer)
                buffer = None
            merged.append(art)
        else:
            # Short article – try to merge
            if buffer is None:
                buffer = {**art}
            elif (buffer["chapter"] == art["chapter"]
                  and buffer["section"] == art["section"]):
                # Same section – merge
                buffer["article_title"] += f" | {art['article_title']}"
                buffer["article_body"] += f"\n\n{art['article_title']}\n{art['article_body']}"
            else:
                # Different section – flush and start new buffer
                merged.append(buffer)
                buffer = {**art}

    if buffer:
        merged.append(buffer)

    return merged


# ============================================================
# 5. MAIN PROCESSING PIPELINE
# ============================================================

def process_legal_pdf(
    pdf_path: str,
    source_file: str,
    law_name: Optional[str] = None,
    domain: Optional[str] = None,
    effective_date: Optional[str] = None,
    law_id: Optional[str] = None,
    max_chunk_len: int = 2800,
    merge_short: bool = True,
    min_merge_len: int = 400,
) -> List[Dict]:

    # Extract & clean
    raw_text = extract_text_with_fitz(pdf_path)
    cleaned_text = clean_legal_pdf_text(raw_text)

    # Parse structure
    articles = parse_structure(cleaned_text)

    # Optionally merge short articles
    if merge_short:
        articles = merge_short_articles(articles, min_merge_len)

    # Build chunks
    final_chunks = []
    chunk_counter = 0

    for art in articles:
        title = art["article_title"]
        body = art["article_body"]
        chapter = art["chapter"]
        section = art.get("section")

        if not body.strip():
            continue

        # Build hierarchical title for retrieval
        hierarchy_parts = [p for p in [title, section, chapter] if p]
        combined_title = " | ".join(hierarchy_parts)

        # Split if needed
        sub_chunks = split_article_by_khoan(title, body, max_chunk_len)

        for idx, chunk_text in enumerate(sub_chunks, start=1):
            chunk_counter += 1
            chunk = {
                "chunk_id": f"{law_id or 'LAW'}_{chunk_counter:04d}",
                "law_name": law_name,
                "law_id": law_id,
                "source_file": source_file,
                "domain": domain,
                "effective_date": effective_date,
                "chapter": chapter,
                "section": section,
                "article": title,
                "title": combined_title,
                "context": chunk_text.strip(),
                "char_count": len(chunk_text.strip()),
            }
            if len(sub_chunks) > 1:
                chunk["sub_part"] = idx
                chunk["total_parts"] = len(sub_chunks)

            final_chunks.append(chunk)

    return final_chunks


# ============================================================
# 6. SAVE
# ============================================================
def save_to_json(data: List[Dict], output_file: str):
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


# ============================================================
# 7. DIAGNOSTICS
# ============================================================
def print_diagnostics(chunks: List[Dict]):
    """Print useful stats about the chunking results."""
    lengths = [c["char_count"] for c in chunks]
    print(f"\n{'='*60}")
    print(f"  CHUNKING DIAGNOSTICS")
    print(f"{'='*60}")
    print(f"  Total chunks    : {len(chunks)}")
    print(f"  Avg length      : {sum(lengths)//len(lengths):,} chars")
    print(f"  Min length      : {min(lengths):,} chars")
    print(f"  Max length      : {max(lengths):,} chars")
    print(f"  Median length   : {sorted(lengths)[len(lengths)//2]:,} chars")

    # Distribution
    brackets = [(0, 300), (300, 800), (800, 1500), (1500, 2800), (2800, 99999)]
    print(f"\n  Length distribution:")
    for lo, hi in brackets:
        count = sum(1 for l in lengths if lo <= l < hi)
        bar = "█" * count
        label = f"{lo}-{hi}" if hi < 99999 else f"{lo}+"
        print(f"    {label:>10s}: {count:3d} {bar}")

    # Articles with sub-parts
    multi = [c for c in chunks if "sub_part" in c]
    if multi:
        arts = set(c["article"] for c in multi)
        print(f"\n  Articles split into sub-parts ({len(arts)}):")
        for art in sorted(arts):
            parts = [c for c in multi if c["article"] == art]
            print(f"    {art[:60]:60s} → {len(parts)} parts")

    print(f"{'='*60}\n")


# ============================================================
# 8. MAIN
# ============================================================
if __name__ == "__main__":
    pdf_path = "E:\VN-legal-chatbot\data_processing\Luat_tieu_dung.pdf"
    source_file = "Luat_BVQLNTD.pdf"
    law_name = "Luật Bảo vệ quyền lợi người tiêu dùng 2023"
    domain = "tieu_dung"
    effective_date = "2024-07-01"
    law_id = "BVQLNTD_2023"
    output_json = "output_optimized_tieu_dung.json"

    chunks = process_legal_pdf(
        pdf_path=pdf_path,
        source_file=source_file,
        law_name=law_name,
        domain=domain,
        effective_date=effective_date,
        law_id=law_id,
        max_chunk_len=2800,
        merge_short=True,
        min_merge_len=400,
    )

    print_diagnostics(chunks)

    # Sample output
    print("Sample chunk (first):")
    print(json.dumps(chunks[0], ensure_ascii=False, indent=2))
    print()
    print("Sample chunk (merged short, if any):")
    for c in chunks:
        if "|" in c.get("article", "") and "Điều" in c["article"].split("|")[1]:
            print(json.dumps(c, ensure_ascii=False, indent=2))
            break

    save_to_json(chunks, output_json)
    print(f"\nSaved {len(chunks)} chunks to {output_json}")

<>:353: SyntaxWarning: invalid escape sequence '\V'
<>:353: SyntaxWarning: invalid escape sequence '\V'
C:\Users\Admin\AppData\Local\Temp\ipykernel_20004\2268178121.py:353: SyntaxWarning: invalid escape sequence '\V'
  pdf_path = "E:\VN-legal-chatbot\data_processing\Luat_tieu_dung.pdf"



  CHUNKING DIAGNOSTICS
  Total chunks    : 83
  Avg length      : 1,275 chars
  Min length      : 308 chars
  Max length      : 2,800 chars
  Median length   : 1,059 chars

  Length distribution:
         0-300:   0 
       300-800:  26 ██████████████████████████
      800-1500:  29 █████████████████████████████
     1500-2800:  27 ███████████████████████████
         2800+:   1 █

  Articles split into sub-parts (5):
    Điều 10. Các hành vi bị nghiêm cấm trong bảo vệ quyền lợi ng → 2 parts
    Điều 3. Giải thích từ ngữ                                    → 2 parts
    Điều 39. Trách nhiệm của tổ chức, cá nhân kinh doanh với ngư → 4 parts
    Điều 77. Trách nhiệm của Ủy ban nhân dân các cấp             → 2 parts
    Điều 8. Bảo vệ quyền lợi người tiêu dùng dễ bị tổn thương    → 2 parts

Sample chunk (first):
{
  "chunk_id": "BVQLNTD_2023_0001",
  "law_name": "Luật Bảo vệ quyền lợi người tiêu dùng 2023",
  "law_id": "BVQLNTD_2023",
  "source_file": "Luat_BVQLNTD.pdf",
  "domain": "tieu